### Addressing Data Skew with Salting

**The Problem:** "Skew" occurs when one data key (e.g., a "Null" or a frequent ID) has significantly more rows than others. Spark sends all rows for one key to a single core, creating a bottleneck which slows down the entire job.<br>

**The Solution:** Adding a Random Salt (an integer or suffix) to the key breaks that one massive group into smaller, manageable chunks that can be processed in parallel across all available cores (6 cores in this setup).

In [1]:
import os
import sys
import time
sys.path.insert(0, os.path.abspath('..'))

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from jobs.tpch_modules import read_tpch_tables,build_enriched_df,introduce_skew,compute_revenue


In [20]:

spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName("TPCH-Salting-Nb")
        .config("spark.executor.instances", "2")
        .config("spark.executor.memory", "512m") \
        .config("spark.executor.cores", "3") \
        .getOrCreate()
    )
spark.conf.set("spark.sql.adaptive.enabled", "false") # this setting when true will auto handle moderate skew.
spark.conf.set("spark.sql.shuffle.partitions", "20") # dont want to burden with sparks default 200 parts!
spark.sparkContext.setLogLevel("ERROR")


In [21]:
NUM_SALTS = 20
base_path = "../data/raw/tpch_sf1"

In [22]:
df_dict = read_tpch_tables(spark,base_path) # Read all the tables.
enriched_df = build_enriched_df(df_dict) # Merge all the tables to make master table.

In [23]:
# Filtering for only one year for this example.
enriched_df = enriched_df.filter(f.col('order_year')==1996)

In [24]:
enriched_rev_df = compute_revenue(enriched_df) # Adds revenue column.
skewed_df = introduce_skew(enriched_rev_df) # Introduces delibrate skew.

In [25]:
def show_partition_distribution(df, title):
    print(f"--- {title} ---")
    # Get the partition ID for every row and count them
    dist = df.withColumn("partition_id", f.spark_partition_id()) \
             .groupBy("partition_id") \
             .count() \
             .orderBy("partition_id")
    dist.show()

In [26]:
df_before = skewed_df.repartition("r_name")
show_partition_distribution(df_before, "Before Salting (Skewed)")

--- Before Salting (Skewed) ---


[Stage 72:=======================================================>(61 + 1) / 62]

+------------+------+
|partition_id| count|
+------------+------+
|           0|    14|
|           2|     5|
|           3|550683|
|           4|    10|
|           7|    12|
|          19|    14|
+------------+------+



### Note:
Even with salting, data may not distribute perfectly evenly.<br>
Multiple salted keys can "collide" and land in the same partition bucket.<br>
Play with the different salt factor (NUM_SALTS) to distribute data evenly to avoid task queuing.

In [27]:
salted_df = skewed_df.withColumn("salt", f.floor(f.rand() * NUM_SALTS).cast("int")) \
             .withColumn("salted_region", f.concat_ws("_",f.col("r_name"), f.col("salt"))) \

df_after = salted_df.repartition("salted_region")
show_partition_distribution(df_after, "After Salting (Balanced)")

--- After Salting (Balanced) ---


[Stage 84:================================================>       (54 + 6) / 62]

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|27829|
|           1|27150|
|           2|27309|
|           4|27360|
|           5|55188|
|           6|54906|
|           7|27660|
|           8|    2|
|           9|55102|
|          10|83009|
|          11|27619|
|          12|    1|
|          13|27670|
|          14|54854|
|          15|27473|
|          16|    9|
|          17|    8|
|          18|27585|
|          19|    4|
+------------+-----+



#### Two-Stage Aggregation: 
For a groupBy to benefit from salting, you must aggregate by the salted_key first, then perform a final aggregation on the original_key.


In [ ]:
partial_agg_df = (
        salted_df.groupBy("r_name", "order_year","salted_region")
            .agg(f.sum("revenue").alias("partial_revenue"))
    )

In [ ]:
# Final Aggregation (Remove Salt)
final_df = (
    partial_agg_df
    .groupBy("r_name", "order_year")
    .agg(f.sum("partial_revenue").alias("total_revenue"))
)
final_df.show()